# Reproduce the pipeline

This notebook reruns the steps of the pipeline in order and checks that every output still matches the committed file.

- By default it reruns only the steps that reproduce the committed outputs exactly (marked "Yes" in `README.md`). Steps 14 and 15 read SciSciNet over the internet.
- Run it from the repository root, for example with `jupyter execute reproduce.ipynb`.
- The scripts overwrite the files in `data/derived/`. If an output differs, the last cell lists the files and raises an error. To restore the committed outputs, run `git checkout -- data/derived`.

In [ ]:
# Steps that cannot reproduce the committed outputs exactly (see README.md). Set to True to rerun them.
RUN_SCRAPING = False  # step 1: scrape the four websites again
RUN_SCOPUS = False    # step 6: needs SCOPUS_API_KEY and access from a network that subscribes to Scopus
RUN_SNAPSHOT = False  # steps 10-13 and a new baseline sample in step 17: needs the OpenAlex snapshot

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DERIVED = ROOT / "data" / "derived"
if not (ROOT / "src").is_dir() or not DERIVED.is_dir():
    raise RuntimeError("Run this notebook from the repository root.")

# Read .env (one KEY=VALUE per line) and pass the values to the scripts
env = os.environ.copy()
if (ROOT / ".env").exists():
    for line in (ROOT / ".env").read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            if value.strip():
                env[key.strip()] = value.strip()

# Keep a copy of the committed outputs to compare with at the end
SAVED = Path(tempfile.mkdtemp(prefix="scisci_saved_")) / "derived"
shutil.copytree(DERIVED, SAVED)
print(f"Copied the committed outputs to {SAVED}")

In [ ]:
SCRAPING_NOTEBOOKS = [
    "notebooks/scraping/toyota_frontier_research_center_scrape.ipynb",
    "notebooks/scraping/toyota_central_rd_labs_scrape.ipynb",
    "notebooks/scraping/aisin_imra_jp_scrape.ipynb",
    "notebooks/scraping/imra_com_scrape.ipynb",
]

# (step, script or notebooks, run it?, extra environment variables)
STEPS = [
    ("1", SCRAPING_NOTEBOOKS, RUN_SCRAPING, {}),
    ("2", "src/rebuild_combined_papers.py", True, {}),
    ("3", "src/openalex_link_toyota_papers.py", True, {}),
    ("4", "src/apply_review_decisions.py", True, {}),
    ("5", "src/build_paper_table.py", True, {}),
    ("6", "src/enrich_papers_scopus.py", RUN_SCOPUS, {}),
    ("7", "src/build_authorship_table.py", True, {}),
    ("8", "src/normalize_affiliations.py", True, {}),
    ("9", "src/aggregate_researchers.py", True, {}),
    ("10", "src/openalex_snapshot_extract.py", RUN_SNAPSHOT, {}),
    ("11", "src/build_reference_titles.py", RUN_SNAPSHOT, {}),
    ("12", "src/build_paper_covariates.py", RUN_SNAPSHOT, {}),
    ("13", "src/novelty_uzzi.py", RUN_SNAPSHOT, {}),
    ("13t", "src/novelty_uzzi_selftest.py", True, {}),
    ("14", "src/fetch_sciscinet_metrics.py", True, {}),
    ("15", "src/verify_novelty_sciscinet.py", True, {}),
    ("16", "src/build_analysis_table.py", True, {}),
    # Without the snapshot, reuse the committed baseline sample and recompute only the percentiles
    ("17", "src/build_matched_baseline.py", True, {} if RUN_SNAPSHOT else {"SCISCI_REUSE_SAMPLE": "1"}),
    ("18", "src/analyze_novelty.py", True, {}),
]


def run(cmd, extra_env):
    start = time.time()
    result = subprocess.run(cmd, cwd=ROOT, env={**env, **extra_env}, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"Failed: {' '.join(cmd)}")
    return time.time() - start


for step, target, enabled, extra in STEPS:
    if not enabled:
        print(f"step {step:>3}: skipped")
        continue
    for t in (target if isinstance(target, list) else [target]):
        if t.endswith(".ipynb"):
            # nbconvert runs the notebook and writes the executed copy to a temporary folder
            cmd = [sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute",
                   "--output-dir", tempfile.mkdtemp(), t]
        else:
            cmd = [sys.executable, t]
        print(f"step {step:>3}: {t}  ({run(cmd, extra):.0f} s)")

In [ ]:
def same(a: Path, b: Path) -> bool:
    if a.suffix == ".csv":
        x, y = pd.read_csv(a), pd.read_csv(b)
        if x.equals(y):
            return True
        try:
            # Allow differences in the last digits of floating-point numbers
            pd.testing.assert_frame_equal(x, y, check_exact=False, rtol=1e-9, atol=1e-12)
            return True
        except AssertionError:
            return False
    if a.suffix == ".parquet":
        return pd.read_parquet(a).equals(pd.read_parquet(b))
    if a.suffix == ".json":
        return json.loads(a.read_text()) == json.loads(b.read_text())
    return a.read_bytes() == b.read_bytes()


differ, missing = [], []
for old in sorted(p for p in SAVED.rglob("*") if p.is_file()):
    rel = old.relative_to(SAVED)
    new = DERIVED / rel
    if not new.exists():
        missing.append(rel)
    elif not same(old, new):
        differ.append(rel)
added = sorted(p.relative_to(DERIVED) for p in DERIVED.rglob("*")
               if p.is_file() and not (SAVED / p.relative_to(DERIVED)).exists())

n = sum(1 for p in SAVED.rglob("*") if p.is_file())
print(f"Compared {n} files: {n - len(differ) - len(missing)} match, {len(differ)} differ, {len(missing)} missing")
for rel in differ:
    print("  differs:", rel)
for rel in missing:
    print("  missing:", rel)
for rel in added:
    print("  new file (not committed):", rel)
if differ or missing:
    raise AssertionError("Some outputs do not match the committed files.")